# 01 · Explore the Upworthy exploratory dataset

## 1. Load data

In [19]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE TABLE packages AS
    SELECT * FROM read_csv_auto('../data/upworthy-archive-exploratory-packages-03.12.2020.csv')
""")

con.execute("SELECT COUNT(*) AS n_rows FROM packages").df()

,n_rows
0,22666


## 2. Columns

In [20]:
con.execute("DESCRIBE packages").df()

,column_name,column_type,null,key,default,extra
0,column00,BIGINT,YES,None,None,None
1,created_at,TIMESTAMP,YES,None,None,None
2,updated_at,TIMESTAMP,YES,None,None,None
3,clickability_test_id,VARCHAR,YES,None,None,None
4,excerpt,VARCHAR,YES,None,None,None
5,headline,VARCHAR,YES,None,None,None
6,lede,VARCHAR,YES,None,None,None
7,slug,VARCHAR,YES,None,None,None
8,eyecatcher_id,VARCHAR,YES,None,None,None
9,impressions,BIGINT,YES,None,None,None


## 3. Example test

In [21]:
example = con.execute("""
    SELECT clickability_test_id,
           headline,
           impressions,
           clicks,
           ROUND(clicks / impressions * 100, 2) AS ctr_pct
    FROM packages
    WHERE clickability_test_id = (SELECT clickability_test_id FROM packages LIMIT 1)
    ORDER BY ctr_pct DESC
""").df()
example

,clickability_test_id,headline,impressions,clicks,ctr_pct
0,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,3052,150,4.91
1,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,3018,132,4.37
2,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,3033,122,4.02
3,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,3092,110,3.56


## 4. Test types

In [22]:
test_types = con.execute("""
    WITH per_test AS (
        SELECT clickability_test_id,
               COUNT(*)                                            AS n_versions,
               COUNT(DISTINCT headline)                            AS n_headlines,
               COUNT(DISTINCT COALESCE(eyecatcher_id, 'none'))      AS n_images,
               COUNT(DISTINCT headline || '|' || COALESCE(eyecatcher_id, 'none')) AS n_combos
        FROM packages
        GROUP BY clickability_test_id
    )
    SELECT *,
           CASE
               WHEN n_combos = 1                        THEN 'A/A'
               WHEN n_headlines > 1 AND n_images = 1    THEN 'headline only'
               WHEN n_headlines = 1 AND n_images > 1    THEN 'image only'
               ELSE 'mixed'
           END AS test_type
    FROM per_test
""").df()

summary = test_types["test_type"].value_counts().to_frame("n_tests")
summary["pct"] = (summary["n_tests"] / summary["n_tests"].sum() * 100).round(1)
summary

,n_tests,pct
test_type,,
headline only,2307,47.3
image only,1426,29.3
A/A,840,17.2
mixed,300,6.2


## 5. Flag the known randomization-problem period
Tests created between 2013-06-25 and 2014-01-10 may have randomization issues (per the archive documentation).

In [23]:
test_dates = con.execute("""
    SELECT clickability_test_id,
           MIN(created_at) AS test_start,
           MIN(created_at) >= TIMESTAMP '2013-06-25'
             AND MIN(created_at) < TIMESTAMP '2014-01-11' AS in_problem_period
    FROM packages
    GROUP BY clickability_test_id
""").df()

print("Date range:", test_dates["test_start"].min(), "to", test_dates["test_start"].max())
test_dates["in_problem_period"].value_counts().to_frame("n_tests")

Date range: 2013-01-26 20:10:36 to 2015-04-29 22:44:51.741000


,n_tests
in_problem_period,
False,3823
True,1050
